<a href="https://colab.research.google.com/github/AaronL123/Flyrank-ML-assignments/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AaronL123/Flyrank-ML-assignments/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

## Two signal checks, before the rule

**Signal 1 — staleness (behind FlyRank's refresh flags): FALSE.**

The refresh flags lean on "how long since this page was last updated." In this warehouse that would come from `last_optimized_date`. It does not survive checking:

- 50.6% of my 59,122 pages have no value at all.
- Of the 29,194 that do, **100% fall after my decision moment** — between 34 and 107 days later (median 82).

So `last_optimized_date` is not a record of when a page was last refreshed; it is forward-looking — a scheduled or planned optimization date. Using it as a staleness feature would pull information from after the decision into the decision itself. **My rule cannot use staleness, and the check is what caught it.** A rule built on the obvious refresh signal would have leaked without ever looking wrong.

**Signal 2 — position tier vs decline (behind the CTR-fix logic): CONFIRMED.**

| Position tier | n | decline rate |
|---|---|---|
| 1–3 | 7,344 | 61.2% |
| 4–10 | 32,068 | 63.4% |
| 11–20 | 10,551 | 70.6% |
| 21–50 | 8,694 | 71.8% |
| 50+ | 445 | 89.7% |

Decline rate rises monotonically with worse position, from 61.2% to 89.7%, against an overall base rate of 65.9%. The relationship is directionally consistent and every bucket has usable n. Median CTR also drops in the 21–50 tier (0.23% against ~0.38% higher up), which is what the CTR-fix logic assumes.

**Caution:** the spread is real but modest across the tiers that hold most of the data — 61.2% to 71.8% covers 58,657 of 59,122 pages. The 89.7% tier is only 445 pages. So position is a usable signal, not a strong one, and a rule leaning on it alone will not separate much.

## The rule

Plain words: **a page is worth reviewing first if it already has real search demand, is ranking outside the top positions, and is converting that demand into clicks poorly.** Demand means someone is looking; weak position and weak CTR mean the page is present but not capturing it.

Reason codes (one per page, first match wins):

- `weak_position_high_demand` — ranks outside the top 10 with substantial impressions
- `low_ctr_good_position` — ranks top 10 but CTR is below the tier median
- `thin_engagement` — has impressions but very few clicks relative to volume

Action labels: `refresh`, `optimize_title_meta`, `review_manually`.

Staleness is deliberately absent — signal 1 ruled it out.

**Caveat, found after building the score:** the individual reason codes do not all beat the base rate. `low_ctr_good_position` (58.5%) and `weak_position_high_demand` (64.5%) score below the 65.9% base rate; only `thin_engagement` (67.8%) beats it. Most strikingly, `no_flag` pages — those my rule scores lowest — have the *highest* decline rate at 80.0%. The aggregate position-tier correlation from signal 2 does not carry cleanly into these compound conditions. This baseline is honestly beatable, and the numbers say by how much.

In [1]:
!pip install -q duckdb

import duckdb, pandas as pd, numpy as np, json, os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN").strip()
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL       = "hf://datasets/FlyRank/internship-warehouse"
CONTENT   = f"{REL}/dim_content.parquet"
DEV_MONTH = f"{REL}/fact_content_daily_performance/month=2026-03/data_0.parquet"
FEAT_END, OUT_START = "2026-03-21", "2026-03-22"

frame = con.sql(f"""
    WITH feat AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)            AS impressions_21d,
               SUM(gsc_clicks)                 AS clicks_21d,
               AVG(NULLIF(gsc_avg_position,0)) AS avg_position_21d,
               COUNT(*)                        AS days_observed
        FROM read_parquet('{DEV_MONTH}')
        WHERE gsc_data_available IS TRUE AND report_date <= DATE '{FEAT_END}'
        GROUP BY 1,2
        HAVING SUM(gsc_clicks) > 0          -- label can only fire where clicks exist
    ),
    outcome AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_clicks) AS clicks_out, COUNT(*) AS days_out
        FROM read_parquet('{DEV_MONTH}')
        WHERE gsc_data_available IS TRUE AND report_date >= DATE '{OUT_START}'
        GROUP BY 1,2
    )
    SELECT f.*, o.clicks_out, o.days_out,
           c.last_optimized_date, c.content_created_date,
           c.content_type, c.search_volume, c.word_count
    FROM feat f
    JOIN outcome o USING (client_hash_id, content_hash_id)
    LEFT JOIN read_parquet('{CONTENT}') c USING (client_hash_id, content_hash_id)
    WHERE o.days_out > 0
""").df()

frame["ctr_21d"]      = frame.clicks_21d / frame.impressions_21d
frame["rate_before"]  = frame.clicks_21d / frame.days_observed
frame["rate_after"]   = frame.clicks_out / frame.days_out
frame["is_declining"] = (frame.rate_after < frame.rate_before).astype(int)
frame["days_since_update"] = (pd.Timestamp("2026-03-21") - pd.to_datetime(frame.last_optimized_date)).dt.days

print(f"Pages: {len(frame):,}   Base rate: {frame.is_declining.mean():.1%}")
print(f"Missing last_optimized_date: {frame.last_optimized_date.isna().mean():.1%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages: 59,122   Base rate: 65.9%
Missing last_optimized_date: 50.6%


In [2]:
# --- Signal 1: staleness (behind FlyRank's refresh flags) ---
s = frame.dropna(subset=["days_since_update"]).copy()
s["bucket"] = pd.cut(s.days_since_update,
                     bins=[-1, 30, 90, 180, 365, 10**6],
                     labels=["0-30d", "31-90d", "91-180d", "181-365d", "365d+"])
t1 = s.groupby("bucket", observed=True).agg(n=("is_declining", "size"),
                                            decline_rate=("is_declining", "mean"))
t1["decline_rate"] = (100 * t1.decline_rate).round(1)
print("SIGNAL 1 — staleness vs decline")
print(f"(usable rows: {len(s):,} of {len(frame):,}; {frame.days_since_update.isna().mean():.1%} have no update date)")
print(t1.to_string())
print(f"overall base rate: {100*frame.is_declining.mean():.1f}%\n")

# --- Signal 2: CTR vs position (behind the CTR-fix logic) ---
p = frame.dropna(subset=["avg_position_21d"]).copy()
p["pos_tier"] = pd.cut(p.avg_position_21d,
                       bins=[0, 3, 10, 20, 50, 10**6],
                       labels=["1-3", "4-10", "11-20", "21-50", "50+"])
t2 = p.groupby("pos_tier", observed=True).agg(n=("is_declining", "size"),
                                              decline_rate=("is_declining", "mean"),
                                              median_ctr=("ctr_21d", "median"))
t2["decline_rate"] = (100 * t2.decline_rate).round(1)
t2["median_ctr"]   = (100 * t2.median_ctr).round(2)
print("SIGNAL 2 — position tier vs decline and CTR")
print(t2.to_string())

SIGNAL 1 — staleness vs decline
(usable rows: 29,194 of 59,122; 50.6% have no update date)
Empty DataFrame
Columns: [n, decline_rate]
Index: []
overall base rate: 65.9%

SIGNAL 2 — position tier vs decline and CTR
              n  decline_rate  median_ctr
pos_tier                                 
1-3        7344          61.2        0.37
4-10      32068          63.4        0.38
11-20     10551          70.6        0.38
21-50      8694          71.8        0.23
50+         445          89.7        0.79


In [3]:
d = frame.days_since_update.dropna()
print(f"n = {len(d):,}")
print(d.describe())
print(f"\nNegative (optimized after 2026-03-21): {(d < 0).sum():,}  ({(d < 0).mean():.1%})")
print(f"Zero or positive: {(d >= 0).sum():,}")

n = 29,194
count    29194.000000
mean       -81.609646
std         15.827310
min       -107.000000
25%        -95.000000
50%        -82.000000
75%        -66.000000
max        -34.000000
Name: days_since_update, dtype: float64

Negative (optimized after 2026-03-21): 29,194  (100.0%)
Zero or positive: 0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
top10_impr = frame.impressions_21d.quantile(0.90)

def assign(row):
    if row.avg_position_21d > 10 and row.impressions_21d >= top10_impr * 0.1:
        return "weak_position_high_demand"
    if row.avg_position_21d <= 10 and row.ctr_21d < 0.0038:
        return "low_ctr_good_position"
    if row.impressions_21d >= 100 and row.ctr_21d < 0.01:
        return "thin_engagement"
    return "no_flag"

frame["reason_code"] = frame.apply(assign, axis=1)

ACTION = {
    "weak_position_high_demand": "refresh",
    "low_ctr_good_position":     "optimize_title_meta",
    "thin_engagement":           "review_manually",
    "no_flag":                   "no_action",
}
frame["action"] = frame.reason_code.map(ACTION)

frame["score"] = (
    frame.impressions_21d.rank(pct=True) * 0.3 +
    frame.avg_position_21d.rank(pct=True) * 0.3 +
    (1 - frame.ctr_21d.rank(pct=True)) * 0.4
)
frame.loc[frame.reason_code == "no_flag", "score"] = 0

queue = frame.sort_values(["score", "content_hash_id"], ascending=[False, True]).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
cols = ["client_hash_id", "content_hash_id", "reason_code", "action", "score",
        "impressions_21d", "avg_position_21d", "ctr_21d", "is_declining"]
queue[cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

base = frame.is_declining.mean()
print(f"Base rate: {base:.1%}")
for k in (20, 50):
    print(f"precision@{k}: {queue.head(k).is_declining.mean():.1%}")

Base rate: 65.9%
precision@20: 60.0%
precision@50: 60.0%


## 3. Top-20 review

Top 10 review — all ten carry `weak_position_high_demand` → `refresh`, ranking positions 40–58 with impressions in the tens of thousands and CTR near zero (0.002–0.016%).

1. Position 56.97, CTR 0.006%. Flagged for huge unclaimed demand at a bad rank. **Wrong because:** not declining — this page's poor CTR may already be its floor, with nothing left to fall.
2. Position 47.40, CTR 0.002%. Same logic, correctly flagged — this one is declining.
3. Position 46.59, CTR 0.007%. Flagged on demand + bad position. **Wrong because:** not declining, and near-identical to row 2 on every feature the rule uses.
4. Position 43.88, CTR 0.008%. Correctly flagged, declining.
5. Position 53.79, CTR 0.016%. **Wrong because:** not declining — highest CTR in the top 10, yet the rule still ranks it above several that are.
6. Position 44.29, CTR 0.016%. Correctly flagged, declining.
7. Position 48.06, CTR 0.008%. Correctly flagged, declining.
8. Position 58.23, CTR 0.011%. Correctly flagged, declining.
9. Position 40.50, CTR 0.002%. **Wrong because:** not declining, despite having the best position in the top 10 — the rule can't use that ordering, since it drove the score up rather than down.
10. Position 39.71, CTR 0.002%. **Wrong because:** not declining, same feature profile as row 9.

**Pattern in the errors:** the five wrong picks (1, 3, 5, 9, 10) are not distinguishable from the five right picks (2, 4, 6, 7, 8) on position or CTR — both groups span the same ranges. The rule scores a *level* (how bad is this page right now); the label is a *change* (did it get worse). A page can be badly ranked and low-CTR while stable, or badly ranked and low-CTR while declining, and nothing in this rule tells the two apart.

In [5]:
review = queue.head(10)[["reason_code", "action", "score",
                          "impressions_21d", "avg_position_21d", "ctr_21d", "is_declining"]]
review.index = range(1, 11)
review

,reason_code,action,score,impressions_21d,avg_position_21d,ctr_21d,is_declining
1,weak_position_high_demand,refresh,0.996876,32201.0,56.969533,0.000062,0
2,weak_position_high_demand,refresh,0.996187,41922.0,47.399855,0.000024,1
3,weak_position_high_demand,refresh,0.995865,43900.0,46.591498,0.000068,0
4,weak_position_high_demand,refresh,0.994941,51869.0,43.877750,0.000077,1
5,weak_position_high_demand,refresh,0.994267,25494.0,53.786967,0.000157,0
6,weak_position_high_demand,refresh,0.993987,45189.0,44.294177,0.000155,1
7,weak_position_high_demand,refresh,0.993903,24791.0,48.056218,0.000081,1
8,weak_position_high_demand,refresh,0.993247,19049.0,58.228089,0.000105,1
9,weak_position_high_demand,refresh,0.992982,41030.0,40.502615,0.000024,0
10,weak_position_high_demand,refresh,0.992452,42185.0,39.712695,0.000024,0


## 4. Weak picks + leakage check

**Weak picks.** The top-10 review shows the core problem: `weak_position_high_demand` scores pages on their current level (bad position, near-zero CTR), but the label is about *change* (did the click rate fall). Rows 1, 3, 5, 9, and 10 were flagged with the same feature profile as several correctly-flagged declining pages — the rule has no way to separate a page that is stably bad from one that is getting worse.

This matches the reason-code decline rates from section 1: `low_ctr_good_position` (58.5%) and `weak_position_high_demand` (64.5%) both sit below the 65.9% base rate, while `no_flag` — pages the rule considers fine — has the highest decline rate of all, 80.0%. A rule scoring only levels cannot see decline; it can only see badness, and badness and decline are not the same thing here.

**What would fix it, for later modeling weeks:** a feature that captures trend within the feature window itself — e.g. comparing the first and second half of the 21-day window — rather than only the window's average level. That is a level-vs-change distinction a hand-written rule can't easily encode, which is exactly the kind of thing a trained model is suited to learn.

**Leakage check.** No product flags (`optimization_eligible_date`, `health_score`, or similar) are used anywhere in the score or reason codes. No column from the outcome window (`clicks_out`, `days_out`, `rate_after`) touches the score — confirmed by inspecting the scoring code, which reads only `impressions_21d`, `avg_position_21d`, and `ctr_21d`, all computed from days 1–21. `last_optimized_date` is not used as a feature, consistent with the FALSE verdict in section 1. `client_hash_id` and `content_hash_id` are used only for grouping and the tiebreak sort, never as score inputs.

In [6]:
SCORE_INPUTS = {"impressions_21d", "avg_position_21d", "ctr_21d"}
BANNED = {"clicks_out", "days_out", "rate_after", "rate_before", "is_declining",
          "last_optimized_date", "days_since_update"}

print("Score built only from:", SCORE_INPUTS)
print("Overlap with banned/label-derived columns:", SCORE_INPUTS & BANNED)
print("→ empty set confirms no leakage into the score" if not (SCORE_INPUTS & BANNED) else "→ LEAK FOUND")

Score built only from: {'impressions_21d', 'ctr_21d', 'avg_position_21d'}
Overlap with banned/label-derived columns: set()
→ empty set confirms no leakage into the score


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.